# Scikit-Learn Module - `model_selection`

## `train_test_split()`

앞서 예제에서 사용했던 `train_test_split()` 함수에 대해 알아보겠습니다.<br>
이 함수를 사용하는 이유는 전체 데이터로 학습했을 때, 그 데이터를 외워버리는 (과적합) 문제가 발생하기 때문입니다.

In [19]:
# 필수 Dependency 설정
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [5]:
# 전체 데이터로 학습하고 전체 데이터를 예측
iris = load_iris()
iris_data = iris.data
iris_target = iris.target

model_dt = DecisionTreeClassifier()

# 전체 데이터로 학습
model_dt.fit(iris_data, iris_target)
# 전체 데이터로 예측
y_pred = model_dt.predict(iris_data)

acc = accuracy_score(iris_target, y_pred)
print(f"Accuracy: {acc:.4f}")
if acc == 1.0:
    print("과적합 발생!")

Accuracy: 1.0000
과적합 발생!


정확도가 1.0이라는 의미는 즉 100% 예측했다는 의미입니다.<br>
그렇다면 이 모델은 모든 붓꽃을 맞출 수 있을까요?<br>
정답은 아닙니다. 전체 데이터로 학습했기 때문에, 전체 데이터를 맞춰보라고 시키면 다 맞추게 됩니다.

때문에 `train_test_split()`을 사용하여 전체 데이터를 외우지 못하게 하는 것입니다.
`train_test_split()`의 주요 인자는 다음과 같습니다.
- `test_size=` : 전체 데이터 셋에서 어느 정도의 비율로 테스트 데이터 셋을 만들지를 정합니다.<br>
    기본값은 0.25, 즉 25%입니다.
- `train_size=` : 전체 데이터에서 어느 정도의 비율로 학습 데이터 셋을 만들지를 정합니다.<br>
    주로 `test_size=`를 사용하기 때문에 잘 사용하지 않습니다.
- `shuffle=` : 데이터 셋의 순서를 섞을 것인지 정합니다.<br>
    주로 데이터 분산이 필요한 학습 데이터 셋에 적용합니다.
- `random_state=` : 무작위 확률로 데이터 셋을 나누기 때문에 같은 결과를 도출하기 위해 Seed를 설정합니다.<br>
    관례로서 42값을 많이 사용합니다.

`train_test_split()`의 return 값은 (학습 데이터셋, 검증 데이터셋, 학습 Label, 검증 Label) 형태의 tuple 형태로 반환됩니다.

In [9]:
# 검증 데이터를 30%, seed를 121로 적용한 예제
model_dt = DecisionTreeClassifier()
X_train, X_val, y_train, y_val = train_test_split(iris_data, iris_target,
                                                  test_size=0.3,
                                                  random_state=121)

model_dt.fit(X_train, y_train)

y_val_pred = model_dt.predict(X_val)
acc = accuracy_score(y_val, y_val_pred)
print(f"Accuracy: {acc:.4f}")
if acc == 1.0:
    print("과적합 발생!")

Accuracy: 0.9556


## 교차검증 (Cross Validation)

그렇다면 `train_test_split()`이 모든 과적합(Overfitting) 문제를 해결할 수 있을까요?<br>
이것도 정답은 아닙니다.<br>
검증 데이터로 Accuracy가 높게 나왔다고 해서 테스트 데이터에서 Accuracy가 낮게 나왔을 때도 과적합을 의심해볼 수 있기 때문입니다.<br>
따라서 이것을 더 명확히 구분하기 위해 교차검증을 사용하게 됩니다.

교차검증은 이러한 데이터 셋의 집합을 여러개 구성하여 학습과 평가를 여러번 수행합니다.<br>
학습데이터에서 다시 학습 / 검증 데이터로 나누어 최종적으로 기존의 검증 데이터 셋으로 평가합니다.

### K-Fold Cross Validation

가장 보편적으로 사용하는 교차검증 기법입니다.<br>
K개의 데이터 셋을 구성하고, K-1번의 학습과 1번의 검증을 수행합니다.

|K   |데이터 셋1|데이터 셋2|데이터 셋3|데이터 셋4|데이터 셋5|결과|
|-----|-------|--------|-------|--------|-------|---|
|1번째|학습|학습|학습|학습|검증|평가1|
|2번째|학습|학습|학습|검증|학습|평가2|
|3번째|학습|학습|검증|학습|학습|평가3|
|4번째|학습|검증|학습|학습|학습|평가4|
|5번째|검증|학습|학습|학습|학습|평가5|

여기서 도출된 평가1~평가5의 평균을 최종 모델 예측성능으로 측정합니다.

In [11]:
# KFold 객체 선언
from sklearn.model_selection import KFold

model_dt = DecisionTreeClassifier(random_state=156)
kf = KFold(n_splits=5)
cv_accuracy = []
print(f"iris data size: {len(iris_data)}")

iris data size: 150


In [18]:
# KFold로 5개 데이터 세트 생성
n_iter = 0

for train_index, val_index in kf.split(iris_data):
    # Fancy Indexing
    X_train, X_val = iris_data[train_index], iris_data[val_index]
    y_train, y_val = iris_target[train_index], iris_target[val_index]

    # 학습 및 예측
    model_dt.fit(X_train, y_train)
    y_val_pred = model_dt.predict(X_val)
    n_iter+=1

    # 반복하는 동안 정확도 저장
    acc = np.round(accuracy_score(y_val, y_val_pred), 4)
    # X_train, X_val 행 수를 가져옴
    train_size = X_train.shape[0]
    val_size = X_val.shape[0]

    print("="*100)
    print(f"K-Fold[{n_iter+1}]\taccuracy:\t{acc}\t학습 데이터셋 크기:\t{train_size}\t검증 데이터셋 크기:\t{val_size}\n")
    print(f"검증 데이터셋 index:\n{val_index}")
    print("="*100)

    cv_accuracy.append(acc)

print(f"K-Fold 평균 정확도: {np.mean(cv_accuracy):.4f}")

K-Fold[2]	accuracy:	1.0	학습 데이터셋 크기:	120	검증 데이터셋 크기:	30

검증 데이터셋 index:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
K-Fold[3]	accuracy:	0.9667	학습 데이터셋 크기:	120	검증 데이터셋 크기:	30

검증 데이터셋 index:
[30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53
 54 55 56 57 58 59]
K-Fold[4]	accuracy:	0.8667	학습 데이터셋 크기:	120	검증 데이터셋 크기:	30

검증 데이터셋 index:
[60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83
 84 85 86 87 88 89]
K-Fold[5]	accuracy:	0.9333	학습 데이터셋 크기:	120	검증 데이터셋 크기:	30

검증 데이터셋 index:
[ 90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119]
K-Fold[6]	accuracy:	0.7333	학습 데이터셋 크기:	120	검증 데이터셋 크기:	30

검증 데이터셋 index:
[120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137
 138 139 140 141 142 143 144 145 146 147 148 149]
K-Fold 평균 정확도: 0.9000


### Stratified K-Fold

Stratified K-Fold는 불균형한(imbalanced) 분포도를 가진 Label 데이터 집합을 위한 K-Fold 방식입니다.<br>
여기서 불균형하다는 의미는 어느 한쪽은 많고, 어느 한쪽을 적게 주어서 불균형한 Label 데이터 셋을 만든다는 의미입니다.

이러한 방식을 사용하는 이유는 K-Fold에도 한계가 있기 때문입니다.

In [20]:
# iris 데이터 셋 분석
iris_features = iris.feature_names
iris_df = pd.DataFrame(data=iris_data, columns=iris_features)
iris_df["label"]=iris_target
iris_df["label"].value_counts()

label
0    50
1    50
2    50
Name: count, dtype: int64

모든 label의 분포도가 50개 씩 분포하고 있습니다.<br>
이 데이터 셋에 label수와 같은 K-Fold(3)를 적용하면 어떻게 되는지 확인해보겠습니다.

In [23]:
# K-Fold의 한계
kf = KFold(n_splits=3)
n_iter = 0
for train_index, val_index in kf.split(iris_df):
    n_iter+=1
    label_train = iris_df["label"].iloc[train_index]
    label_val = iris_df["label"].iloc[val_index]

    print("="*100)
    print(f"[K-Fold {n_iter}]")
    print(f"Train label count:\n{label_train.value_counts()}")
    print(f"Val label count:\n{label_val.value_counts()}")

[K-Fold 1]
Train label count:
label
1    50
2    50
Name: count, dtype: int64
Val label count:
label
0    50
Name: count, dtype: int64
[K-Fold 2]
Train label count:
label
0    50
2    50
Name: count, dtype: int64
Val label count:
label
1    50
Name: count, dtype: int64
[K-Fold 3]
Train label count:
label
0    50
1    50
Name: count, dtype: int64
Val label count:
label
2    50
Name: count, dtype: int64


- K-Fold 1<br>
학습 Label: 1, 2만 학습<br>
검증 Label: 0만 검증
- K-Fold 2<br>
학습 Label: 0, 2만 학습<br>
검증 Label: 1만 검증
- K-Fold 3<br>
학습 Label: 0, 1만 학습<br>
검증 Label: 2만 검증

각 폴드별 모델은 검증 Label을 학습하지 못하므로, 정확도는 모두 0이 됩니다.

`StratifiedKFold`는 Label의 불균형을 주는 방식이므로, 이런 데이터 분포에 대응할 수 있게 됩니다.<br>
Label에 대한 정보가 필요하기 때문에, `KFold`와 달리 Label의 정보를 추가해서 넣어주어야 합니다.

In [24]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3)
n_iter = 0

for train_index, val_index in skf.split(iris_df, iris_df["label"]):
    n_iter+=1
    label_train = iris_df["label"].iloc[train_index]
    label_val = iris_df["label"].iloc[val_index]

    print("="*100)
    print(f"[K-Fold {n_iter}]")
    print(f"Train label count:\n{label_train.value_counts()}")
    print(f"Val label count:\n{label_val.value_counts()}")

[K-Fold 1]
Train label count:
label
2    34
0    33
1    33
Name: count, dtype: int64
Val label count:
label
0    17
1    17
2    16
Name: count, dtype: int64
[K-Fold 2]
Train label count:
label
1    34
0    33
2    33
Name: count, dtype: int64
Val label count:
label
0    17
2    17
1    16
Name: count, dtype: int64
[K-Fold 3]
Train label count:
label
0    34
1    33
2    33
Name: count, dtype: int64
Val label count:
label
1    17
2    17
0    16
Name: count, dtype: int64


모든 Fold의 Label들이 고르게 섞인 것을 확인할 수 있습니다.

In [29]:
# 붓꽃 데이터에 StratifiedKFold 사용
model_dt = DecisionTreeClassifier(random_state=42)
skf = StratifiedKFold(n_splits=3)
cv_accuracy = []
n_iter = 0

for train_index, val_index in skf.split(iris_df, iris_df["label"]):
    n_iter+=1
    X_train, X_val = iris_data[train_index], iris_data[val_index]
    y_train, y_val = iris_target[train_index], iris_target[val_index]

    model_dt.fit(X_train, y_train)
    y_val_pred = model_dt.predict(X_val)

    acc = np.round(accuracy_score(y_val, y_val_pred), 4)
    cv_accuracy.append(acc)

    train_size = X_train.shape[0]
    val_size = X_val.shape[0]

    divider = "="*50
    print(f"StratifiedKFold[{n_iter}]\t{divider}")
    print(f"acc:\t{acc:.4f}\ttrain size:\t{train_size}\tval size:\t{val_size}")
print("\n")
print(f"StratifiedKFold accuracy: {cv_accuracy}")
print(f"StratifiedKFold accuracy mean: {np.mean(cv_accuracy):.4f}")

StratifiedKFold[1]	==================================================
acc:	0.9800	train size:	100	val size:	50
StratifiedKFold[2]	==================================================
acc:	0.9400	train size:	100	val size:	50
StratifiedKFold[3]	==================================================
acc:	0.9600	train size:	100	val size:	50


StratifiedKFold accuracy: [0.98, 0.94, 0.96]
StratifiedKFold accuracy mean: 0.9600


일반적으로 분류 문제에서의 교차검증은 K-Fold 대신 Stratified K-Fold를 사용하는 것이 좋습니다.<br>
하지만 회귀문제에서는 Label의 범위가 수치형이기 때문에 Stratified K-Fold를 사용할 수 없습니다.

### 교차검증을 간단하게 적용하기 - `cross_val_score()`

위에서 K-Fold를 적용하는 Flow는 다음과 같았습니다.
1. Fold Dataset 생성
2. 반복문으로 Fold 순회
3. 반복해서 학습 및 예측 수행

`cross_val_score()`는 이 Flow를 한번에 실행하는 함수입니다.<br>
`cross_val_score()`는 Estimator가 분류에 해당한다면 자동으로 `StratifiedKFold`를 사용합니다.<br>
주요 인자는 다음과 같습니다.
- `estimator=` : 분류/회귀 모델 클래스
- `X=` : 전체 데이터셋
- `y=` : label 데이터 (분류일 때만 사용 가능)
- `scoring=`: 집계 방식
- `cv=` : Fold 수

In [30]:
# cross_val_score 사용 예제
from sklearn.model_selection import cross_val_score

model_dt = DecisionTreeClassifier(random_state=42)

scores = cross_val_score(model_dt, iris_data, iris_target, cv=3, scoring="accuracy")
print(f"Accuracy: {np.round(scores, 4)}")
print(f"Accuracy mean: {np.round(np.mean(scores), 4)}")

Accuracy: [0.98 0.94 0.96]
Accuracy mean: 0.96


더 많은 정보를 제공하는 `cross_validate()` 함수도 존재합니다.<br>
여기서 추가로 학습 데이터에 대한 평가지표와 수행시간 까지 같이 제공합니다.<br>
하지만 대부분의 경우 `cross_val_score()`로 충분하기 때문에 자세히 설명하지 않겠습니다.

In [39]:
# cross_validate 사용 예제
from sklearn.model_selection import cross_validate

model_dt = DecisionTreeClassifier(random_state=42)

validations = cross_validate(model_dt, iris_data, iris_target, cv=3)

valid_fit_time = validations["fit_time"]
valid_score_time = validations["score_time"]
valid_scores = validations["test_score"]

for index in range(len(valid_scores)):
    print(f"fit_time: {valid_fit_time[index]:.4f}\tscore_time: {valid_score_time[index]:.4f}\tscore: {valid_scores[index]:.4f}")
print("\n")
print(f"fit time mean: {np.mean(valid_fit_time * 1000):.4f}ms")
print(f"score time mean: {np.mean(np.array(valid_score_time) * 1000):.4f}ms")
print(f"score mean: {np.mean(np.array(valid_scores)):.4f}")

fit_time: 0.0010	score_time: 0.0003	score: 0.9800
fit_time: 0.0009	score_time: 0.0002	score: 0.9400
fit_time: 0.0006	score_time: 0.0002	score: 0.9600


fit time mean: 0.8293ms
score time mean: 0.2523ms
score mean: 0.9600


## 교차검증 + 하이퍼파라미터 튜닝 - GridSearchCV

하이퍼파라미터는 모델이 직접 정할 수 없는 값들을 의미합니다.<br>
즉, 사람이 직접 조정해줘야 하는 값입니다.

`GridSearchCV`의 Grid는 격자 형태로 파라미터를 나열해서 촘촘하게 입력해 테스트하는 방식에서 붙은 이름입니다.<br>
파라미터를 순차적으로 변경하면서 최적화가 가능하다는 의미가 됩니다.

In [40]:
# Grid Parameter 예시
grid_parameters = {
    "max_depth": [1, 2, 3],
    "min_samples_split": [2, 3]
}

위의 코드에서 `GridSearchCV`를 돌린다면 3 * 2의 6가지를 경우의 수를 cv 만큼 수행하게 됩니다.<br>
즉 `cv=3`이라면 3 * 2 * 3 = 18번 수행합니다.

`GridSearchCv`의 주요 인자는 다음과 같습니다.
- `estimator=`
- `param_grid=` : 딕셔너리를 받으며, `{파라미터명: 파라미터값}` 형태로 지정합니다.
- `scoring=`
- `cv=`
- `refit=` : model을 최적의 파라미터값으로 적용합니다. 기본값은 True입니다.

In [46]:
# GridSearchCV 사용 예제
from sklearn.model_selection import GridSearchCV

X_train, X_val, y_train, y_val = train_test_split(iris_data, iris_target, test_size=0.2, random_state=121)
model_dt = DecisionTreeClassifier()

# GridSearchCV용 파라미터 선언
parameters = {
    "max_depth": [1, 2, 3],
    "min_samples_split": [2, 3]
}

# GridSearchCV 초기화
grid_cv_dt = GridSearchCV(model_dt, parameters, cv=3)

grid_cv_dt.fit(X_train, y_train)

# GridSearchCV 결과를 추출해 DataFrame으로 변환
scores_df = pd.DataFrame(grid_cv_dt.cv_results_)
scores_cols = ["params", "mean_test_score", "rank_test_score", "split0_test_score", "split1_test_score", "split2_test_score"]
scores_df[scores_cols]

,params,mean_test_score,rank_test_score,split0_test_score,split1_test_score,split2_test_score
0,"{'max_depth': 1, 'min_samples_split': 2}",0.700000,5,0.700,0.7,0.70
1,"{'max_depth': 1, 'min_samples_split': 3}",0.700000,5,0.700,0.7,0.70
2,"{'max_depth': 2, 'min_samples_split': 2}",0.958333,3,0.925,1.0,0.95
3,"{'max_depth': 2, 'min_samples_split': 3}",0.958333,3,0.925,1.0,0.95
4,"{'max_depth': 3, 'min_samples_split': 2}",0.975000,1,0.975,1.0,0.95
5,"{'max_depth': 3, 'min_samples_split': 3}",0.975000,1,0.975,1.0,0.95


총 6개의 결과 값이 출력되었습니다.
- `mean_test_score` : 파라미터 별 교차검증의 평균 정확도
- `rank_test_score` : 파라미터 별 정확도 순위
- `splitN_test_score` : cv당 교차검증 정확도

여기서 rank가 1인 parameter는 2개인데 어떤 값이 선정되었는지 확인하려면 다음과 같이 출력합니다.

In [47]:
# GridSearchCV의 최적값 확인
print(f"GridSearchCV Best Parameter:\t{grid_cv_dt.best_params_}")
print(f"GridSearchCV Best Accuracy: \t{grid_cv_dt.best_score_:.4f}")

GridSearchCV Best Parameter:	{'max_depth': 3, 'min_samples_split': 2}
GridSearchCV Best Accuracy: 	0.9750


최적의 Parameter가 적용된 Estimator를 불러와서 예측을 수행하려면 다음과 같이 출력합니다.

In [48]:
# 최적 Parameter가 적용된 모델 사용
best_estimator = grid_cv_dt.best_estimator_

# 이미 학습이 되어있으므로 학습을 시킬 필요가 없습니다.
y_val_pred = best_estimator.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")

Validation Accuracy: 0.9667
